# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane 2 — Refresh / Content Opportunity Scoring.**

I'm picking this lane because I've already run the starter pipeline on `content_refresh_anonymized.csv`, so I have real, verified evidence that a learned ranking beats a fixed hand-rule at this exact task (Precision@50 goes from 0.240 with the baseline rule to 0.740 with a random forest — see Section 3). It also produces the cleanest decision-action-cost story of the available lanes: a content editor with limited time each sprint needs a ranked list of pages to review first, and that's a problem this data can directly support without needing causal claims about whether a refresh actually works.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Unit of analysis:** one content page (`content_id`), described by its trailing 90-day search and engagement signals (impressions, clicks, sessions, CTR, average position, age, freshness).

**The question:** given limited review capacity, which pages should a content editor look at first for refresh, expansion, or pruning?

**The decision:** which pages make it into this sprint's review queue, out of thousands of candidates.

**The action:** an editor pulls the top N pages from the ranked output and rewrites, expands, or updates them (or flags them for monitoring/pruning if that's the recommended action).

**The cost of a wrong call:**
- False positive (page flagged, not actually a good use of time) → editor hours spent rewriting a page that wasn't really at risk or opportunity.
- False negative (page not flagged, but actually declining with real demand) → a page that matters keeps losing visibility, unreviewed, until the next cycle.

Because editor time is the scarce resource here, precision at the *top* of the ranked queue matters more than catching every possible page — which is why Precision@K is the right metric, not overall accuracy.

**Why data/ML helps at all:** with 30,000+ pages and 44 signals each, no reviewer can eyeball which pages combine "declining" + "still has real demand" + "worth the rewrite effort." A transparent rule-based baseline already exists (see Section 3), but the starter run shows a model finds combinations of signal the fixed rule misses — a ~3x lift in precision at the top of the queue.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [1]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("total pages in starter dataset:", len(df))

# same eligibility filter the starter pipeline uses
eligible = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
print(f"eligible pages (impressions_90d>0 & age>=90d): {len(eligible)} ({len(eligible)/len(df):.1%})")

# how much of the eligible pool looks like a real refresh candidate?
declining_with_demand = eligible[(eligible["trend_direction"] == "down") & (eligible["impressions_90d"] >= 100)]
print(f"'declining_with_demand' candidates: {len(declining_with_demand)} ({len(declining_with_demand)/len(eligible):.1%} of eligible)")

stale_visible = eligible[(eligible["days_since_last_update"] >= 180) & (eligible["impressions_90d"] >= 500)]
print(f"'stale_visible_page' candidates: {len(stale_visible)} ({len(stale_visible)/len(eligible):.1%} of eligible)")

print()
print("trend_direction breakdown:")
print(eligible["trend_direction"].value_counts())


total pages in starter dataset: 30000
eligible pages (impressions_90d>0 & age>=90d): 30000 (100.0%)
'declining_with_demand' candidates: 13152 (43.8% of eligible)
'stale_visible_page' candidates: 17 (0.1% of eligible)

trend_direction breakdown:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim:**
- This ranks pages by *observed* signals (search volume, position, CTR, engagement, trend) as candidates for editorial review.
- It's *decision-support*: it helps a limited-capacity review team prioritize where to spend time first.
- Any lift over the baseline rule is *measured* against a held-out set of clients the model never saw during training (client-holdout validation).

**What I cannot claim:**
- That a refresh will *cause* a page to recover — that requires a real experiment (e.g. before/after with a control group), not a ranking model.
- Anything about Google's actual ranking algorithm — I only have observable outcomes (impressions, clicks, position), not the mechanism behind them.
- That "declining" pages found here are the only pages worth reviewing, or that the model's top pick is guaranteed better than the 51st pick — precision@K is directional, not a certainty per page.

I'll keep language in the rest of this project to *observed*, *measured*, *directional*, and *decision-support* — never "predicts" or "proves."


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.